#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.cust_info")

In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos

def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()
    
    return

In [0]:
info(df)

#Transformations

## Rename columns' names

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Unique Values

### customer_id, customer_number"

In [0]:
#Explorando os valores duplicados

id = df.groupby("customer_id").count()
id.filter(col("count") > 1).display()
lista_id = [row.customer_id for row in id.select("customer_id").filter(col("count") > 1).collect()]

print(lista_id)
df.filter(col("customer_id").isin(lista_id)).display()

print("Os Ids repetidos podem ser apagados, pois temos um dos duplicados com o cadastro completo. Porém só apagar os duplicados vazios")

In [0]:
#Apagando os duplicados vazios
columns_to_check = [
    "first_name",
    "last_name",    
]

print(lista_id)

condition = (
    (col("customer_id").isin(lista_id) == True )
    & (sum([col(c).isNull().cast("int") for c in columns_to_check]) > 0)
    )

df_clean = df.filter(condition == False)
df_clean = df_clean.dropDuplicates(["customer_id"])
info(df_clean)
df_clean.filter(col("customer_id").isin(lista_id)).display()




## Missings Values

In [0]:
df_clean = df_clean.fillna("Unknown", subset=["gender"])
df_clean.groupBy("gender").count().display()
info(df_clean)

## Normalization

In [0]:
df_clean = df_clean.withColumn("gender", when(col("gender") == "M", "Male")
                               .when(col("gender") == "M", "Male").otherwise("Unknown"))
df_clean = df_clean.withColumn("marital_status", when(col("marital_status") == "M", "Married")
                               .when(col("marital_status") == "S", "Singe").otherwise("Unknown"))
df_clean.display()

## Trimm

In [0]:
for field in df_clean.schema.fields:
    if isinstance(field.dataType, StringType):
        df_clean = df_clean.withColumn(field.name, trim(col(field.name)))

In [0]:
display(df_clean.limit(20))

#Write into Silver Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.crm_customers")
